# Validación visual de detección de edificios - Fase 1

**Tarea 1.10**

Para cada polígono procesado, muestra las 9 imágenes Sentinel-2 anuales en una grilla 3x3 con overlay de los edificios detectados (coloreados según su `fecha_aparicion`).

Este notebook es el control humano del pipeline automático: permite verificar que el algoritmo no cuenta árboles/suelo desnudo como edificios y que las fechas de aparición son plausibles.

Fecha: 2026-04-22

In [ ]:
# Imports
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, get_cmap
from rasterio.plot import reshape_as_image

In [ ]:
# Cargar datasets
RUTA_BUILDINGS = Path('../data/raw/google_buildings/posadas_buildings.geojson')
RUTA_EDIFICIOS_CONTEO = Path('../data/processed/conteos/edificios_con_fecha.csv')

buildings_gdf = gpd.read_file(RUTA_BUILDINGS) if RUTA_BUILDINGS.exists() else None
conteo_df = (
    pd.read_csv(RUTA_EDIFICIOS_CONTEO) if RUTA_EDIFICIOS_CONTEO.exists() else None
)

if conteo_df is None:
    print(
        'No se encontró edificios_con_fecha.csv. '
        'Corré scripts/20_contar_techos.py primero.'
    )
else:
    print(f'Edificios con fecha: {len(conteo_df)}')
    print(conteo_df['fecha_aparicion'].value_counts().head(15))

In [ ]:
# Parámetros: elegir un polígono a validar
POLIGONO_ID = 'itaembe_mini'  # cambiar para validar otro
RUTA_SENTINEL_RGB = Path('../data/raw/sentinel2')

# Las 9 fechas target declaradas en settings.yaml
FECHAS = [
    '201807', '201907', '202007', '202107', '202207',
    '202307', '202407', '202507', '202607',
]

# Edificios del polígono
if conteo_df is not None:
    subset = conteo_df[conteo_df['poligono_id'] == POLIGONO_ID].copy()
    print(f'Edificios en polígono {POLIGONO_ID}: {len(subset)}')
else:
    subset = pd.DataFrame(columns=['lat', 'lon', 'fecha_aparicion'])

In [ ]:
# Helper: cargar imagen RGB desde GeoTIFF multi (bandas B4, B3, B2)
def cargar_rgb(ruta_tif: Path) -> np.ndarray | None:
    if not ruta_tif.exists():
        return None
    with rasterio.open(ruta_tif) as ds:
        if ds.count < 3:
            return None
        # Orden de bandas en multi: B2, B3, B4, B8, B11, B12 -> RGB necesita B4,B3,B2
        r = ds.read(3)  # B4
        g = ds.read(2)  # B3
        b = ds.read(1)  # B2
        rgb = np.stack([r, g, b], axis=-1).astype(np.float32)
        # Escalado simple para Sentinel (reflectancia 0-3000)
        rgb = np.clip(rgb / 3000.0, 0.0, 1.0)
        transform = ds.transform
        bounds = ds.bounds
    return rgb, transform, bounds

In [ ]:
# Grid 3x3 con overlay de edificios coloreados por fecha de aparición (colormap viridis)
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle(
    f'Validación visual - {POLIGONO_ID} (2018-2026)',
    fontsize=16,
    fontweight='bold',
)

cmap = get_cmap('viridis')
norm = Normalize(vmin=2018, vmax=2026)

def fecha_a_anio_float(fecha: str) -> float | None:
    if fecha == '<2018':
        return 2017.5
    if fecha == 'desconocida':
        return None
    try:
        yyyy, mm = fecha.split('-')
        return int(yyyy) + (int(mm) - 1) / 12.0
    except Exception:
        return None

for i, yyyymm in enumerate(FECHAS):
    ax = axes.flat[i]
    yyyy_mm = f'{yyyymm[:4]}-{yyyymm[4:6]}'
    ruta = RUTA_SENTINEL_RGB / f'{POLIGONO_ID}_{yyyymm}_multi.tif'
    resultado = cargar_rgb(ruta)
    if resultado is None:
        ax.set_title(f'{yyyy_mm}\n(sin imagen)')
        ax.axis('off')
        continue
    rgb, _, bounds = resultado
    extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)
    ax.imshow(rgb, extent=extent)

    # Overlay: edificios cuya fecha_aparicion <= fecha del subplot
    fecha_subplot = yyyy_mm
    visibles = subset[
        (subset['fecha_aparicion'] != 'desconocida')
        & (
            (subset['fecha_aparicion'] == '<2018')
            | (subset['fecha_aparicion'] <= fecha_subplot)
        )
    ]
    if len(visibles):
        anios = visibles['fecha_aparicion'].apply(fecha_a_anio_float)
        colores = cmap(norm(anios.fillna(2017.5)))
        ax.scatter(
            visibles['lon'],
            visibles['lat'],
            c=colores,
            s=8,
            edgecolors='white',
            linewidths=0.2,
            alpha=0.85,
        )
    ax.set_title(f'{yyyy_mm}  (n={len(visibles)})')
    ax.set_xticks([])
    ax.set_yticks([])

# Colorbar global
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
fig.colorbar(
    sm,
    ax=axes,
    orientation='horizontal',
    fraction=0.04,
    pad=0.04,
    label='Año de aparición',
)
plt.show()

In [ ]:
# Tabla resumen: conteo por año vs. ground truth visual (usuario completa dict)
GROUND_TRUTH_VISUAL = {
    # 'YYYY-MM': count_manual_estimado_por_el_usuario,
    '2018-07': None,
    '2021-07': None,
    '2024-07': None,
}

if len(subset):
    filas = []
    for yyyymm in FECHAS:
        fecha = f'{yyyymm[:4]}-{yyyymm[4:6]}'
        n_sistema = int(
            (
                (subset['fecha_aparicion'] == '<2018')
                | (
                    (subset['fecha_aparicion'] != 'desconocida')
                    & (subset['fecha_aparicion'] <= fecha)
                )
            ).sum()
        )
        filas.append(
            {
                'fecha': fecha,
                'n_sistema': n_sistema,
                'n_manual': GROUND_TRUTH_VISUAL.get(fecha),
            }
        )
    resumen = pd.DataFrame(filas)
    resumen['delta_pct'] = (
        100.0 * (resumen['n_sistema'] - resumen['n_manual']) / resumen['n_manual']
    ).round(1)
    print(resumen.to_string(index=False))
else:
    print('Sin datos de conteo para comparar.')

In [ ]:
# Export del notebook a HTML para adjuntar al reporte metodológico
# Ejecutar desde la carpeta notebooks/
import subprocess

OUTPUT_DIR = Path('../data/outputs').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    subprocess.run(
        [
            'jupyter',
            'nbconvert',
            '--to',
            'html',
            '--output-dir',
            str(OUTPUT_DIR),
            '01_validacion_fase1.ipynb',
        ],
        check=False,
    )
    print(f'Export a HTML disparado hacia {OUTPUT_DIR}.')
except FileNotFoundError:
    print('jupyter nbconvert no disponible en este entorno.')

## Checklist de validación

Al revisar la grilla 3x3, chequear:

- [ ] ¿El algoritmo está contando árboles/cobertura verde como edificios? (si hay muchos puntos sobre áreas verdes, sí).
- [ ] ¿Hay edificios obvios (manchas claras con sombra rectangular) que NO están siendo detectados?
- [ ] ¿La fecha de aparición es plausible? Un edificio aparece en el año X cuando aún no se ve en el año X-1.
- [ ] ¿El conteo del polígono de control consolidado (Villa Cabello) muestra baja variación interanual?
- [ ] ¿El conteo del polígono de crecimiento rápido (Itaembé Miní/Guazú) muestra tendencia creciente monotónica?
- [ ] ¿Algún año tiene cobertura de nubes que contamine la detección? Marcar en warnings de `scripts/20_contar_techos.py`.

Si algún ítem falla sistemáticamente, ajustar `ndbi_threshold`, `ndvi_threshold` o `confidence_min` en la CLI del script de conteo.